In [4]:
import heapq
import itertools

INITIAL = (0, 1, 2,
           3, 4, 5,
           6, 7, 8)
GOAL = (5, 4, 0,
        6, 1, 8,
        7, 3, 2)

def misplaced_tiles(state, goal):
    return sum(1 for s, g in zip(state, goal) if s != 0 and s != g)


def manhattan_distance(state, goal):
    pos = {tile: divmod(i, 3) for i, tile in enumerate(goal)}
    total = 0
    for i, tile in enumerate(state):
        if tile != 0:
            r, c = divmod(i, 3)
            gr, gc = pos[tile]
            total += abs(r - gr) + abs(c - gc)
    return total

def get_neighbors(state):
    b = state.index(0)
    r, c = divmod(b, 3)
    for dr, dc, move in ((-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n = nr * 3 + nc
            new = list(state)
            new[b], new[n] = new[n], new[b]
            yield tuple(new), move


def is_solvable(state, goal):
    def inversions(s):
        t = [x for x in s if x != 0]
        return sum(1 for i in range(len(t)) for j in range(i + 1, len(t)) if t[i] > t[j])
    return inversions(state) % 2 == inversions(goal) % 2


def print_state(state):
    for i in range(0, 9, 3):
        print(" ".join(str(x) if x else "_" for x in state[i:i + 3]))

def a_star(start, goal, h):
    counter = itertools.count()
    start_h = h(start, goal)
    open_list = [(start_h, start_h, next(counter), 0, start)]
    best_g = {start: 0}
    parent = {start: (None, None)}
    closed = set()
    expanded = 0

    while open_list:
        f, _, _, g, state = heapq.heappop(open_list)
        if state in closed:
            continue
        closed.add(state)
        expanded += 1

        if state == goal:
            path, moves = [], []
            while state is not None:
                path.append(state)
                state, mv = parent[state]
                if mv:
                    moves.append(mv)
            return path[::-1], moves[::-1], expanded, len(best_g)

        for nxt, mv in get_neighbors(state):
            ng = g + 1
            if nxt not in best_g or ng < best_g[nxt]:
                best_g[nxt] = ng
                parent[nxt] = (state, mv)
                nh = h(nxt, goal)
                heapq.heappush(open_list, (ng + nh, nh, next(counter), ng, nxt))
    return None, None, expanded, len(best_g)


def run_case(name, h, start, goal):
    print("=" * 50)
    print(name)
    print("=" * 50)
    path, moves, expanded, generated = a_star(start, goal, h)
    if path is None:
        print("No solution found.")
        return
    for step, s in enumerate(path):
        print(f"\nStep {step}: g = {step}, h = {h(s, goal)}, f = {step + h(s, goal)}"
              + (f" (move blank {moves[step - 1]})" if step else " (initial state)"))
        print_state(s)
    print(f"\nSolution depth (moves) : {len(moves)}")
    print(f"Moves : {" -> ".join(moves)}")
    print(f"Nodes expanded : {expanded}")
    print(f"Nodes generated : {generated}\n")


if __name__ == "__main__":
    start, goal = INITIAL, GOAL

    if not is_solvable(start, goal):
        print("This puzzle is not solvable.")
    else:
        run_case("CASE 1: g(n) = depth, h(n) = number of misplaced tiles",
                 misplaced_tiles, start, goal)
        run_case("CASE 2: g(n) = depth, h(n) = Manhattan distance",
                 manhattan_distance, start, goal)

CASE 1: g(n) = depth, h(n) = number of misplaced tiles

Step 0: g = 0, h = 8, f = 8 (initial state)
_ 1 2
3 4 5
6 7 8

Step 1: g = 1, h = 8, f = 9 (move blank Right)
1 _ 2
3 4 5
6 7 8

Step 2: g = 2, h = 7, f = 9 (move blank Down)
1 4 2
3 _ 5
6 7 8

Step 3: g = 3, h = 7, f = 10 (move blank Right)
1 4 2
3 5 _
6 7 8

Step 4: g = 4, h = 7, f = 11 (move blank Up)
1 4 _
3 5 2
6 7 8

Step 5: g = 5, h = 8, f = 13 (move blank Left)
1 _ 4
3 5 2
6 7 8

Step 6: g = 6, h = 8, f = 14 (move blank Left)
_ 1 4
3 5 2
6 7 8

Step 7: g = 7, h = 8, f = 15 (move blank Down)
3 1 4
_ 5 2
6 7 8

Step 8: g = 8, h = 8, f = 16 (move blank Right)
3 1 4
5 _ 2
6 7 8

Step 9: g = 9, h = 8, f = 17 (move blank Right)
3 1 4
5 2 _
6 7 8

Step 10: g = 10, h = 8, f = 18 (move blank Up)
3 1 _
5 2 4
6 7 8

Step 11: g = 11, h = 8, f = 19 (move blank Left)
3 _ 1
5 2 4
6 7 8

Step 12: g = 12, h = 8, f = 20 (move blank Left)
_ 3 1
5 2 4
6 7 8

Step 13: g = 13, h = 7, f = 20 (move blank Down)
5 3 1
_ 2 4
6 7 8

Step 14: g = 14, 